# Model Development & Training

This notebook covers the development and training of the sentiment analysis model using transfer learning.

## Objectives
- Load pre-trained HuggingFace model
- Configure training parameters
- Implement fine-tuning pipeline
- Log experiments with MLflow
- Save model artifacts

## 1. Setup & Imports

In [ ]:
import os
import sys
import yaml
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import mlflow
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 2. Load Configuration

In [ ]:
# TODO: Load parameters from YAML
with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

train_params = params['train']
data_params = params['data']

print("Training Parameters:")
for key, value in train_params.items():
    print(f"  {key}: {value}")

## 3. Load Data

In [ ]:
# TODO: Load preprocessed training data
train_path = '../data/processed/train.pkl'
test_path = '../data/processed/test.pkl'

print(f"Loading training data from {train_path}...")

if Path(train_path).exists() and Path(test_path).exists():
    with open(train_path, 'rb') as f:
        train_data = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)
    print(f"Loaded {len(train_data)} training samples")
    print(f"Loaded {len(test_data)} test samples")
else:
    print("ERROR: Preprocessed data not found!")
    print("TODO: Run src/data_loader.py to preprocess data")
    
    # Create sample data for demonstration
    train_data = pd.DataFrame({
        'text': ['This is great', 'This is bad'],
        'label': [1, 0]
    })
    test_data = pd.DataFrame({
        'text': ['Good movie', 'Bad movie'],
        'label': [1, 0]
    })

print(f"\nTrain data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

## 4. Load Model & Tokenizer

In [ ]:
# TODO: Load pre-trained model and tokenizer
model_name = train_params['model_name']
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print(f"Model loaded successfully")
print(f"Model size: {model.num_parameters():,} parameters")

## 5. Tokenize Data

In [ ]:
# TODO: Convert to HuggingFace Dataset and tokenize
print("Tokenizing data...")

train_dataset = Dataset.from_pandas(train_data[['text', 'label']])
test_dataset = Dataset.from_pandas(test_data[['text', 'label']])

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        max_length=train_params['max_length'],
        truncation=True,
        padding=True
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# Remove text column and rename label column
train_tokenized = train_tokenized.remove_columns('text')
test_tokenized = test_tokenized.remove_columns('text')

print(f"Tokenization complete")
print(f"Sample tokenized input: {train_tokenized[0]}")

## 6. Define Training Arguments

In [ ]:
# TODO: Set up training arguments
training_args = TrainingArguments(
    output_dir='../results',
    num_train_epochs=train_params['epochs'],
    per_device_train_batch_size=train_params['batch_size'],
    per_device_eval_batch_size=train_params['batch_size'],
    warmup_steps=train_params.get('warmup_steps', 500),
    weight_decay=train_params.get('weight_decay', 0.01),
    learning_rate=float(train_params['learning_rate']),
    logging_dir='../logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    seed=42,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    dataloader_num_workers=0
)

print("Training arguments configured")

## 7. Define Metrics

In [ ]:
# TODO: Define evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0)
    }

print("Metrics function defined")

## 8. Train Model

In [ ]:
# TODO: Start MLflow run and train
mlflow.set_experiment('sentiment-analysis')

with mlflow.start_run():
    # Log parameters
    mlflow.log_params({
        'model_name': model_name,
        'batch_size': train_params['batch_size'],
        'epochs': train_params['epochs'],
        'learning_rate': train_params['learning_rate'],
        'max_length': train_params['max_length']
    })
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=test_tokenized,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer),
    )
    
    # Train the model
    print("Starting training...")
    train_result = trainer.train()
    
    # Get training metrics
    print(f"\nTraining completed!")
    print(f"Training loss: {train_result.training_loss:.4f}")

## 9. Evaluate Model

In [ ]:
# TODO: Evaluate on test set
print("Evaluating model on test set...")
eval_results = trainer.evaluate(eval_dataset=test_tokenized)

print("\nTest Set Results:")
for key, value in eval_results.items():
    if key != 'epoch':
        print(f"  {key}: {value:.4f}")

# Log metrics to MLflow
mlflow.log_metrics({
    'test_accuracy': eval_results['eval_accuracy'],
    'test_f1': eval_results['eval_f1'],
    'test_precision': eval_results['eval_precision'],
    'test_recall': eval_results['eval_recall']
})

## 10. Save Model

In [ ]:
# TODO: Save model and tokenizer
model_save_path = '../models/trained'
Path(model_save_path).mkdir(parents=True, exist_ok=True)

# Save model
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Also save as pickle for compatibility
with open(f'{model_save_path}/model.pkl', 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved to {model_save_path}")

# Log model to MLflow
mlflow.transformers.log_model(model, 'model', tokenizer=tokenizer)
print("Model logged to MLflow")

## 11. Summary & Next Steps

In [ ]:
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)

summary = f"""
Model Configuration:
  - Model: {model_name}
  - Epochs: {train_params['epochs']}
  - Batch Size: {train_params['batch_size']}
  - Learning Rate: {train_params['learning_rate']}
  - Max Length: {train_params['max_length']}

Training Results:
  - Training Loss: {train_result.training_loss:.4f}
  - Test Accuracy: {eval_results['eval_accuracy']:.4f}
  - Test F1 Score: {eval_results['eval_f1']:.4f}
  - Test Precision: {eval_results['eval_precision']:.4f}
  - Test Recall: {eval_results['eval_recall']:.4f}

Artifacts Saved:
  - Model: {model_save_path}
  - MLflow Run: Check MLflow UI

Next Steps:
  1. View metrics in MLflow: mlflow ui
  2. Run evaluation script: python src/evaluate.py
  3. Test inference: python src/inference.py
  4. Deploy API: python app/api.py
"""

print(summary)